In [15]:

import pickle
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

np.set_printoptions(precision=6, suppress=True)

print("Libraries imported successfully.")


Libraries imported successfully.


In [16]:

# CELL 1 — IMPORTS AND SETTINGS
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

np.set_printoptions(precision=6, suppress=True)

MODEL_FILE = Path("geometric_start.pkl")
if not MODEL_FILE.exists():
    raise FileNotFoundError(
        "geometric_start.pkl not found. Run the original FEM notebook through "
        "its export cell and place geometric_start.pkl in this notebook's folder."
    )

with MODEL_FILE.open("rb") as f:
    DATA = pickle.load(f)

print("Imports and model file ready.")


Imports and model file ready.


In [17]:

# CELL 2 — LOAD THE EXISTING STRUCTURAL MODEL
required_keys = [
    "FEM_MODEL", "ETYPE", "DOF_PER_NODE", "NODE_IDS",
    "ORIGINAL_TOTAL_DOF", "TOTAL_DOF", "ANALYSIS_SEGMENTS",
    "K_GLOBAL", "F_GLOBAL", "DISPLACEMENTS", "C_CONSTRAINT",
    "MEMBER_FORCES"
]
missing = [k for k in required_keys if k not in DATA]
if missing:
    raise KeyError(f"Exported model is missing keys: {missing}")

FEM_MODEL = DATA["FEM_MODEL"]
ETYPE = str(DATA["ETYPE"]).upper()
DOF_PER_NODE = int(DATA["DOF_PER_NODE"])
NODE_IDS = list(DATA["NODE_IDS"])
ORIGINAL_TOTAL_DOF = int(DATA["ORIGINAL_TOTAL_DOF"])
TOTAL_DOF = int(DATA["TOTAL_DOF"])
ANALYSIS_SEGMENTS = list(DATA["ANALYSIS_SEGMENTS"])

K_LINEAR = np.asarray(DATA["K_GLOBAL"], dtype=float)
F_EXTERNAL = np.asarray(DATA["F_GLOBAL"], dtype=float).reshape(-1)
U_LINEAR = np.asarray(DATA["DISPLACEMENTS"], dtype=float).reshape(-1)
C_CONSTRAINT = np.asarray(DATA["C_CONSTRAINT"], dtype=float)
MEMBER_FORCES = DATA.get("MEMBER_FORCES", {})

nodes = FEM_MODEL["nodes"]
elements = FEM_MODEL["elements"]
E = float(FEM_MODEL["E"])
A = float(FEM_MODEL["A"])
I = FEM_MODEL.get("I", None)
I = float(I) if I is not None else None

if ETYPE not in {"TRUSS", "BEAM", "FRAME"}:
    raise ValueError(f"Unsupported element type: {ETYPE}")
if DOF_PER_NODE != (2 if ETYPE == "TRUSS" else 3):
    raise ValueError("DOF_PER_NODE does not match ETYPE.")
if K_LINEAR.shape != (TOTAL_DOF, TOTAL_DOF):
    raise ValueError("K_GLOBAL dimensions do not match TOTAL_DOF.")
if F_EXTERNAL.size != TOTAL_DOF or U_LINEAR.size != TOTAL_DOF:
    raise ValueError("Load/displacement vector dimensions do not match TOTAL_DOF.")
if C_CONSTRAINT.ndim != 2 or C_CONSTRAINT.shape[1] != TOTAL_DOF:
    raise ValueError("C_CONSTRAINT dimensions do not match TOTAL_DOF.")
if E <= 0 or A <= 0:
    raise ValueError("E and A must be positive.")
if ETYPE in {"BEAM", "FRAME"} and (I is None or I <= 0):
    raise ValueError("BEAM/FRAME analysis requires positive I.")

print("Model loaded.")
print("Element type:", ETYPE)
print("Nodes:", len(NODE_IDS), "| Segments:", len(ANALYSIS_SEGMENTS))
print("Total DOFs:", TOTAL_DOF, "| K:", K_LINEAR.shape, "| F:", F_EXTERNAL.shape)
print("E =", E, "| A =", A, "| I =", I)


Model loaded.
Element type: TRUSS
Nodes: 4 | Segments: 6
Total DOFs: 8 | K: (8, 8) | F: (8,)
E = 200000000000.0 | A = 0.0001 | I = None


In [18]:

# CELL 3 — VALIDATE SEGMENTS AND EXTRACT INITIAL AXIAL FORCES
# Corrected extraction: use DATA, not the undefined geometric_data variable.
INITIAL_AXIAL_FORCES = {}

for seg in ANALYSIS_SEGMENTS:
    sid = seg["segment_id"]
    eid = seg["original_eid"]
    records = MEMBER_FORCES.get(eid, [])
    if isinstance(records, dict):
        records = [records]
    rec = next((r for r in records if r.get("segment_id") == sid), None)
    if rec is None and len(records) == 1:
        rec = records[0]
    if rec is None:
        print(f"Warning: no initial force record found for segment {sid}.")
        continue

    lf = np.asarray(rec.get("local_force", []), dtype=float).reshape(-1)
    # Local end-force ordering in the supplied model:
    # TRUSS: [N_i, V_i, N_j, V_j] only if length 4; FRAME/BEAM: [N_i,V_i,M_i,N_j,V_j,M_j].
    if ETYPE == "TRUSS":
        if lf.size < 3:
            raise ValueError(f"Segment {sid}: invalid truss local-force vector.")
        Nj = float(lf[2])
    else:
        if lf.size < 6:
            raise ValueError(f"Segment {sid}: invalid frame/beam local-force vector.")
        Nj = float(lf[3])
    INITIAL_AXIAL_FORCES[sid] = Nj
    print(f"Segment {sid} (element {eid}): initial axial force = {Nj:.6f}")

print("Initial axial-force extraction complete.")


Segment 1 (element 1): initial axial force = 0.000000
Segment 2 (element 2): initial axial force = -5382.524219
Segment 3 (element 3): initial axial force = 3806.019375
Segment 4 (element 4): initial axial force = 1688.543593
Segment 5 (element 5): initial axial force = -0.000000
Segment 6 (element 6): initial axial force = -11193.980625
Initial axial-force extraction complete.


In [19]:

# CELL 4 — CONSTRAINT NULL SPACE
def constraint_nullspace(C, ndof):
    C = np.asarray(C, dtype=float)
    if C.shape[0] == 0:
        return np.eye(ndof)
    _, s, Vt = np.linalg.svd(C, full_matrices=True)
    tol = max(C.shape) * np.finfo(float).eps * (s[0] if s.size else 1.0)
    rank = int(np.sum(s > tol))
    Z = Vt[rank:].T
    if not np.allclose(C @ Z, 0.0, atol=1e-8):
        raise RuntimeError("Could not construct constraint null space.")
    return Z

Z = constraint_nullspace(C_CONSTRAINT, TOTAL_DOF)
if Z.shape[1] == 0:
    raise ValueError("All DOFs are constrained.")
print("Free generalized DOFs:", Z.shape[1])
print("Constraint-nullspace error:", np.linalg.norm(C_CONSTRAINT @ Z))


Free generalized DOFs: 3
Constraint-nullspace error: 9.957992501029599e-17


In [20]:

# CELL 5 — MEMBER GEOMETRY AND DOF HELPERS
def segment_geometry(seg):
    xi, yi = float(seg["xi"]), float(seg["yi"])
    xj, yj = float(seg["xj"]), float(seg["yj"])
    dx, dy = xj - xi, yj - yi
    L0 = float(np.hypot(dx, dy))
    if L0 <= 1e-12:
        raise ValueError(f"Segment {seg['segment_id']} has zero initial length.")
    theta0 = float(np.arctan2(dy, dx))
    return np.array([xi, yi]), np.array([xj, yj]), L0, theta0

for seg in ANALYSIS_SEGMENTS:
    expected = 4 if ETYPE == "TRUSS" else 6
    dofs = np.asarray(seg["dofs"], dtype=int)
    if dofs.size != expected:
        raise ValueError(f"Segment {seg['segment_id']} has {dofs.size} DOFs; expected {expected}.")
    if np.any(dofs < 0) or np.any(dofs >= TOTAL_DOF):
        raise ValueError(f"Segment {seg['segment_id']} contains invalid global DOFs.")

print("Segment DOF and geometry checks passed.")


Segment DOF and geometry checks passed.


In [21]:

# CELL 6 — COROTATIONAL ELEMENT INTERNAL FORCE
# Tension is positive. The chord rotation alpha is updated from the current end coordinates.
# TRUSS: axial energy only.
# BEAM/FRAME: axial energy + standard two-end rotational bending energy in the chord frame.

def element_internal_force_global(seg, u):
    dofs = np.asarray(seg["dofs"], dtype=int)
    ue = np.asarray(u, dtype=float)[dofs]
    pi, pj, L0, theta0 = segment_geometry(seg)

    if ETYPE == "TRUSS":
        ui = ue[[0, 1]]
        uj = ue[[2, 3]]
        ci = pi + ui
        cj = pj + uj
        chord = cj - ci
        L = float(np.linalg.norm(chord))
        if L <= 1e-12:
            raise ValueError(f"Segment {seg['segment_id']} collapsed to near-zero length.")
        n = chord / L
        N = E * A / L0 * (L - L0)
        f = np.zeros(4)
        f[0:2] = -N * n
        f[2:4] = N * n
        return dofs, f

    ui = ue[[0, 1]]
    uj = ue[[3, 4]]
    ci = pi + ui
    cj = pj + uj
    chord = cj - ci
    L = float(np.linalg.norm(chord))
    if L <= 1e-12:
        raise ValueError(f"Segment {seg['segment_id']} collapsed to near-zero length.")
    alpha = float(np.arctan2(chord[1], chord[0]))
    n = chord / L

    # Axial response
    N = E * A / L0 * (L - L0)
    f = np.zeros(6)
    f[[0, 1]] -= N * n
    f[[3, 4]] += N * n

    # Bending response from end rotations relative to the current chord.
    phi_i = ue[2] - (alpha - theta0)
    phi_j = ue[5] - (alpha - theta0)
    kb = (E * I / L0) * np.array([[4.0, 2.0], [2.0, 4.0]])
    moments = kb @ np.array([phi_i, phi_j])
    Mi, Mj = float(moments[0]), float(moments[1])
    f[2] += Mi
    f[5] += Mj

    # Translational contribution from the rotation of the chord (energy derivative).
    # d(alpha)/d[x_i,y_i,x_j,y_j] = [dy/L^2, -dx/L^2, -dy/L^2, dx/L^2].
    dx, dy = chord
    grad_alpha = np.array([dy/(L*L), -dx/(L*L), -dy/(L*L), dx/(L*L)])
    f[[0, 1, 3, 4]] -= (Mi + Mj) * grad_alpha

    return dofs, f


def assemble_internal_force(u):
    fint = np.zeros(TOTAL_DOF)
    for seg in ANALYSIS_SEGMENTS:
        dofs, fe = element_internal_force_global(seg, u)
        fint[dofs] += fe
    return fint

print("Corotational internal-force functions defined.")


Corotational internal-force functions defined.


In [22]:

# CELL 7 — NUMERICAL TANGENT STIFFNESS
# Central finite differences of internal force; symmetrized to suppress numerical noise.
def assemble_tangent(u, fd_step=1e-7):
    u = np.asarray(u, dtype=float)
    Kt = np.zeros((TOTAL_DOF, TOTAL_DOF), dtype=float)
    for j in range(TOTAL_DOF):
        h = fd_step * max(1.0, abs(u[j]))
        up = u.copy(); up[j] += h
        um = u.copy(); um[j] -= h
        Kt[:, j] = (assemble_internal_force(up) - assemble_internal_force(um)) / (2.0*h)
    return 0.5 * (Kt + Kt.T)

# Sanity check: internal force at zero displacement should be zero for unloaded initial geometry.
f0 = assemble_internal_force(np.zeros(TOTAL_DOF))
print("Zero-displacement internal-force norm:", np.linalg.norm(f0))


Zero-displacement internal-force norm: 0.0


In [23]:

# CELL 8 — ITERATIVE CHORD-CORRECTION SOLVER
def run_chord_correction(
    n_steps=5,
    max_iterations=40,
    displacement_tolerance=1e-9,
    residual_tolerance=1e-7,
    relaxation=1.0,
    fd_step=1e-7,
    verbose=True,
):
    if n_steps < 1:
        raise ValueError("n_steps must be >= 1.")
    if max_iterations < 1:
        raise ValueError("max_iterations must be >= 1.")
    if not (0.0 < relaxation <= 1.0):
        raise ValueError("relaxation must be in (0, 1].")

    u = np.zeros(TOTAL_DOF, dtype=float)
    history = []
    all_converged = True

    for step in range(1, n_steps + 1):
        lam = step / n_steps
        Fstep = lam * F_EXTERNAL
        converged = False
        du_norm = np.inf
        r_norm = np.inf

        for it in range(1, max_iterations + 1):
            fint = assemble_internal_force(u)
            residual = Fstep - fint
            rr = Z.T @ residual
            r_norm = float(np.linalg.norm(rr))
            scale_r = max(1.0, float(np.linalg.norm(Z.T @ Fstep)))

            if r_norm <= residual_tolerance * scale_r:
                du_norm = 0.0
                converged = True
                break

            Kt = assemble_tangent(u, fd_step=fd_step)
            Kr = Z.T @ Kt @ Z
            if not np.all(np.isfinite(Kr)):
                raise FloatingPointError("Non-finite reduced tangent matrix.")
            if np.linalg.matrix_rank(Kr) < Kr.shape[0]:
                raise np.linalg.LinAlgError(
                    f"Reduced tangent is singular/unstable at step {step}, iteration {it}."
                )

            dq = np.linalg.solve(Kr, rr)
            du = Z @ dq
            du_norm = float(np.linalg.norm(du))
            scale_u = max(1.0, float(np.linalg.norm(u + du)))
            u = u + relaxation * du

            # Check displacement increment and re-evaluate equilibrium residual.
            fint_new = assemble_internal_force(u)
            r_new = Z.T @ (Fstep - fint_new)
            r_norm = float(np.linalg.norm(r_new))
            scale_r = max(1.0, float(np.linalg.norm(Z.T @ Fstep)))

            if (du_norm <= displacement_tolerance * scale_u and
                    r_norm <= residual_tolerance * scale_r):
                converged = True
                break

        history.append({
            "step": step,
            "load_factor": lam,
            "iterations": it,
            "converged": converged,
            "displacement_increment_norm": du_norm,
            "reduced_residual_norm": r_norm,
        })
        all_converged = all_converged and converged

        if verbose:
            print(
                f"Step {step}/{n_steps} | lambda={lam:.3f} | "
                f"iterations={it} | du={du_norm:.3e} | "
                f"reduced residual={r_norm:.3e} | converged={converged}"
            )

        if not converged:
            print("WARNING: a load step did not converge; stopping continuation.")
            break

    return {
        "displacements": u,
        "history": history,
        "all_converged": all_converged,
        "final_load_factor": history[-1]["load_factor"] if history else 0.0,
        "internal_force": assemble_internal_force(u),
        "tangent_stiffness": assemble_tangent(u, fd_step=fd_step),
    }

# Adjust these settings if needed.
N_LOAD_STEPS = 5
MAX_ITERATIONS = 40
DISPLACEMENT_TOLERANCE = 1e-9
RESIDUAL_TOLERANCE = 1e-7
RELAXATION = 1.0

CHORD_RESULT = run_chord_correction(
    n_steps=N_LOAD_STEPS,
    max_iterations=MAX_ITERATIONS,
    displacement_tolerance=DISPLACEMENT_TOLERANCE,
    residual_tolerance=RESIDUAL_TOLERANCE,
    relaxation=RELAXATION,
    fd_step=1e-7,
)

D_FINAL = CHORD_RESULT["displacements"]
print("All load steps converged:", CHORD_RESULT["all_converged"])


Step 1/5 | lambda=0.200 | iterations=3 | du=0.000e+00 | reduced residual=1.232e-08 | converged=True
Step 2/5 | lambda=0.400 | iterations=3 | du=0.000e+00 | reduced residual=1.015e-08 | converged=True
Step 3/5 | lambda=0.600 | iterations=3 | du=0.000e+00 | reduced residual=1.458e-08 | converged=True
Step 4/5 | lambda=0.800 | iterations=3 | du=0.000e+00 | reduced residual=7.065e-09 | converged=True
Step 5/5 | lambda=1.000 | iterations=3 | du=0.000e+00 | reduced residual=1.325e-08 | converged=True
All load steps converged: True


In [24]:

# CELL 9 — FINAL NODAL DISPLACEMENTS AND UPDATED CHORD GEOMETRY
U_NODES_FINAL = D_FINAL[:ORIGINAL_TOTAL_DOF].reshape(len(NODE_IDS), DOF_PER_NODE)

nodal_rows = []
for i, nid in enumerate(NODE_IDS):
    vals = U_NODES_FINAL[i]
    row = {"node": nid, "Ux": vals[0], "Uy": vals[1]}
    if DOF_PER_NODE == 3:
        row["Rz_rad"] = vals[2]
    nodal_rows.append(row)
NODAL_DISPLACEMENTS = pd.DataFrame(nodal_rows)

print("FINAL NODAL DISPLACEMENTS")
display(NODAL_DISPLACEMENTS)

CHORD_GEOMETRY_FINAL = []
for seg in ANALYSIS_SEGMENTS:
    pi, pj, L0, theta0 = segment_geometry(seg)
    dofs = np.asarray(seg["dofs"], dtype=int)
    ue = D_FINAL[dofs]
    if ETYPE == "TRUSS":
        ui, uj = ue[0:2], ue[2:4]
    else:
        ui, uj = ue[0:2], ue[3:5]
    chord = (pj + uj) - (pi + ui)
    L = float(np.linalg.norm(chord))
    theta = float(np.arctan2(chord[1], chord[0]))
    dtheta = float(np.arctan2(np.sin(theta-theta0), np.cos(theta-theta0)))
    CHORD_GEOMETRY_FINAL.append({
        "segment_id": seg["segment_id"],
        "original_eid": seg["original_eid"],
        "L0": L0,
        "L_final": L,
        "extension": L-L0,
        "chord_rotation_rad": dtheta,
        "chord_rotation_deg": np.degrees(dtheta),
    })

CHORD_GEOMETRY_FINAL = pd.DataFrame(CHORD_GEOMETRY_FINAL)
print("FINAL CHORD GEOMETRY")
display(CHORD_GEOMETRY_FINAL)


FINAL NODAL DISPLACEMENTS


,node,Ux,Uy
0,1,0.000000,0.000000e+00
1,2,0.000000,0.000000e+00
2,3,0.002694,-5.982566e-19
3,4,0.003646,-2.800256e-03


FINAL CHORD GEOMETRY


,segment_id,original_eid,L0,L_final,extension,chord_rotation_rad,chord_rotation_deg
0,1,1,5.000000,5.000000,0.000000e+00,0.000000,0.000000
1,2,2,7.071068,7.069163,-1.904907e-03,-0.000270,-0.015441
2,3,3,5.000000,5.000952,9.521973e-04,-0.000560,-0.032082
3,4,4,7.071068,7.071667,5.993035e-04,-0.000645,-0.036930
4,5,5,5.000000,5.000001,7.259297e-07,-0.000539,-0.030874
5,6,6,5.000000,4.997201,-2.798926e-03,-0.000730,-0.041800


In [25]:

# CELL 10 — LOCAL MEMBER END FORCES
# Reconstruct local end-force vectors from corotational internal forces.
# For BEAM/FRAME: [N_i, V_i, M_i, N_j, V_j, M_j].
# For TRUSS: [N_i, V_i, N_j, V_j].
#
# Equivalent local UDL vectors are subtracted where a UDL is specified in FEM_MODEL['loads'].
# This cell supports the UDL schema used in the source model (target='MEMBER',
# type='UDL', element=<element id>, magnitude, angle, start, end). Other load
# types are reported as not reconstructed in the member-force output.

def local_transformation(theta, etype):
    c, s = np.cos(theta), np.sin(theta)
    if etype == "TRUSS":
        return np.array([[c, s, 0, 0],
                         [-s, c, 0, 0],
                         [0, 0, c, s],
                         [0, 0, -s, c]], dtype=float)
    return np.array([
        [c, s, 0, 0, 0, 0],
        [-s, c, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 0, c, s, 0],
        [0, 0, 0, -s, c, 0],
        [0, 0, 0, 0, 0, 1],
    ], dtype=float)

def equivalent_local_udl(seg, theta, L):
    feq = np.zeros(4 if ETYPE == "TRUSS" else 6)
    if ETYPE == "TRUSS":
        return feq
    supported = {"UDL", "UNIFORM", "UNIFORMLY DISTRIBUTED LOAD"}
    for load in FEM_MODEL.get("loads", []):
        if str(load.get("target", "")).upper() != "MEMBER":
            continue
        if int(load.get("element", -999)) != int(seg["original_eid"]):
            continue
        ltype = str(load.get("type", "")).upper()
        if ltype not in supported:
            continue
        mag = float(load.get("magnitude", 0.0))
        angle = np.deg2rad(float(load.get("angle", 90.0)))
        # The original model stores magnitudes in kN/m while FEM vectors are N.
        qx_global, qy_global = 1000.0*mag*np.cos(angle), 1000.0*mag*np.sin(angle)
        c, s = np.cos(theta), np.sin(theta)
        qx_local = c*qx_global + s*qy_global
        qy_local = -s*qx_global + c*qy_global
        start = float(load.get("start", 0.0))
        end = float(load.get("end", seg.get("L", L)))
        if start > 1e-9 or abs(end - L) > 1e-6:
            print(f"Warning: partial UDL on element {seg['original_eid']} is not yet handled.")
            continue
        feq += np.array([
            qx_local*L/2.0,
            qy_local*L/2.0,
            qy_local*L*L/12.0,
            qx_local*L/2.0,
            qy_local*L/2.0,
            -qy_local*L*L/12.0,
        ])
    return feq

MEMBER_END_FORCES = {}
for seg in ANALYSIS_SEGMENTS:
    sid = seg["segment_id"]
    dofs, fint_e = element_internal_force_global(seg, D_FINAL)
    pi, pj, L0, theta0 = segment_geometry(seg)
    if ETYPE == "TRUSS":
        ue = D_FINAL[dofs]
        chord = (pj + ue[2:4]) - (pi + ue[0:2])
    else:
        ue = D_FINAL[dofs]
        chord = (pj + ue[3:5]) - (pi + ue[0:2])
    Lcur = float(np.linalg.norm(chord))
    theta_cur = float(np.arctan2(chord[1], chord[0]))
    Tcur = local_transformation(theta_cur, ETYPE)
    flocal = Tcur @ fint_e

    if ETYPE == "TRUSS":
        # Use the local force components in the two end translation DOFs.
        end_force = flocal.copy()
        MEMBER_END_FORCES[sid] = {
            "segment_id": sid,
            "original_eid": seg["original_eid"],
            "L_final": Lcur,
            "local_end_force": end_force,
            "axial_force": float(E*A/L0*(Lcur-L0)),
            "note": "Corotational deformation-derived force; member-load recovery is not implemented for truss."
        }
    else:
        feq = equivalent_local_udl(seg, theta_cur, Lcur)
        end_force = flocal - feq
        MEMBER_END_FORCES[sid] = {
            "segment_id": sid,
            "original_eid": seg["original_eid"],
            "L_final": Lcur,
            "local_end_force": end_force,
            "axial_force": float(E*A/L0*(Lcur-L0)),
            "note": "Corotational force; full recovery is provided for full-span UDLs only."
        }

force_rows = []
for sid, rec in MEMBER_END_FORCES.items():
    f = rec["local_end_force"]
    row = {"segment_id": sid, "original_eid": rec["original_eid"], "L_final": rec["L_final"]}
    if ETYPE == "TRUSS":
        row.update({"N_i": f[0], "V_i": f[1], "N_j": f[2], "V_j": f[3]})
    else:
        row.update({"N_i": f[0], "V_i": f[1], "M_i": f[2],
                    "N_j": f[3], "V_j": f[4], "M_j": f[5]})
    force_rows.append(row)

MEMBER_FORCE_TABLE = pd.DataFrame(force_rows)
print("FINAL LOCAL MEMBER END FORCES")
display(MEMBER_FORCE_TABLE)


FINAL LOCAL MEMBER END FORCES


,segment_id,original_eid,L_final,N_i,V_i,N_j,V_j
0,1,1,5.000000,0.000000,0.000000e+00,0.000000,0.000000e+00
1,2,2,7.069163,5387.890600,-4.547474e-13,-5387.890600,4.547474e-13
2,3,3,5.000952,-3808.789245,-4.440892e-16,3808.789245,4.440892e-16
3,4,4,7.071667,-1695.086139,3.410605e-13,1695.086139,-3.410605e-13
4,5,5,5.000001,-2.903719,-5.204170e-17,2.903719,5.204170e-17
5,6,6,4.997201,11195.704370,-7.656098e-13,-11195.704370,7.656098e-13


In [26]:

# CELL 11 — CONVERGENCE, EQUILIBRIUM CHECKS, AND SAVE RESULTS
F_FINAL = CHORD_RESULT["final_load_factor"] * F_EXTERNAL
fint_final = assemble_internal_force(D_FINAL)
residual = fint_final - F_FINAL
reduced_residual = Z.T @ residual
constraint_error = C_CONSTRAINT @ D_FINAL

print("CONVERGENCE HISTORY")
display(pd.DataFrame(CHORD_RESULT["history"]))
print("All load steps converged:", CHORD_RESULT["all_converged"])
print("Final load factor:", CHORD_RESULT["final_load_factor"])
print("Reduced equilibrium residual norm:", np.linalg.norm(reduced_residual))
print("Constraint violation norm:", np.linalg.norm(constraint_error))

output = {
    "method": "Iterative corotational chord-update analysis",
    "ETYPE": ETYPE,
    "load_factor": CHORD_RESULT["final_load_factor"],
    "displacements": D_FINAL,
    "nodal_displacements": NODAL_DISPLACEMENTS.to_dict(orient="records"),
    "chord_geometry": CHORD_GEOMETRY_FINAL.to_dict(orient="records"),
    "member_end_forces": MEMBER_END_FORCES,
    "history": CHORD_RESULT["history"],
    "all_converged": CHORD_RESULT["all_converged"],
    "reduced_residual_norm": float(np.linalg.norm(reduced_residual)),
    "constraint_error_norm": float(np.linalg.norm(constraint_error)),
    "method_note": (
        "Corotational chord-update solver with numerical tangent. "
        "Validate on benchmark structures before design use. Full member-load "
        "recovery is currently implemented only for full-span UDLs on beam/frame elements."
    ),
}
with open("chord_correction_results.pkl", "wb") as f:
    pickle.dump(output, f)

NODAL_DISPLACEMENTS.to_csv("chord_nodal_displacements.csv", index=False)
CHORD_GEOMETRY_FINAL.to_csv("chord_geometry_results.csv", index=False)
MEMBER_FORCE_TABLE.to_csv("chord_member_end_forces.csv", index=False)

print("Saved:")
print(" - chord_correction_results.pkl")
print(" - chord_nodal_displacements.csv")
print(" - chord_geometry_results.csv")
print(" - chord_member_end_forces.csv")


CONVERGENCE HISTORY


,step,load_factor,iterations,converged,displacement_increment_norm,reduced_residual_norm
0,1,0.2,3,True,0.0,1.232343e-08
1,2,0.4,3,True,0.0,1.014717e-08
2,3,0.6,3,True,0.0,1.457673e-08
3,4,0.8,3,True,0.0,7.064864e-09
4,5,1.0,3,True,0.0,1.325134e-08


All load steps converged: True
Final load factor: 1.0
Reduced equilibrium residual norm: 1.3251342312773947e-08
Constraint violation norm: 2.682990247114995e-19
Saved:
 - chord_correction_results.pkl
 - chord_nodal_displacements.csv
 - chord_geometry_results.csv
 - chord_member_end_forces.csv


In [27]:
# ============================================================
# COMPARISON: CHORD CORRECTION vs GEOMETRIC STIFFNESS
# ============================================================

import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# 1. Load results from both methods
chord_file = Path("chord_correction_results.pkl")
geom_file  = Path("geometric_results.pkl")

if not chord_file.exists() or not geom_file.exists():
    raise FileNotFoundError(
        "Both result files must exist in this notebook's folder.\n"
        "Run both analysis notebooks and save their results first."
    )

with chord_file.open("rb") as f:
    chord = pickle.load(f)

with geom_file.open("rb") as f:
    geom = pickle.load(f)

# 2. Check load factors and convergence
print("=" * 65)
print("CHORD CORRECTION vs GEOMETRIC STIFFNESS")
print("=" * 65)

print("Chord correction converged:", chord["all_converged"])
print("Geometric stiffness converged:", geom["all_converged"])
print("Chord load factor:", chord["load_factor"])
print("Geometric load factor:",
      geom.get("history", [{}])[-1].get("load_factor", np.nan))

if not chord["all_converged"] or not geom["all_converged"]:
    print("WARNING: At least one method did not fully converge.")

# 3. Compare final displacement vectors
u_cc = np.asarray(chord["displacements"], dtype=float).ravel()
u_gs = np.asarray(geom["displacements"], dtype=float).ravel()

if u_cc.shape != u_gs.shape:
    raise ValueError(
        f"DOF mismatch: chord={u_cc.shape}, geometric={u_gs.shape}. "
        "Check that both analyses use the same model and DOF ordering."
    )

du = u_cc - u_gs
disp_abs_error = np.linalg.norm(du)
disp_reference = np.linalg.norm(u_gs)

disp_rel_error = (
    100.0 * disp_abs_error / disp_reference
    if disp_reference > 1e-14 else np.nan
)

print("\n--- NODAL DISPLACEMENT COMPARISON ---")
print(f"Chord displacement norm:     {np.linalg.norm(u_cc):.6e}")
print(f"Geometric displacement norm: {np.linalg.norm(u_gs):.6e}")
print(f"Absolute difference norm:    {disp_abs_error:.6e}")
print(f"Relative difference:         {disp_rel_error:.4f}%")
print(f"Maximum absolute DOF error:  {np.max(np.abs(du)):.6e}")

# 4. Compare member axial forces using common segment IDs
cc_forces = chord.get("member_end_forces", {})
gs_forces = geom.get("axial_forces", {})

force_rows = []

for sid in sorted(set(cc_forces) & set(gs_forces), key=str):
    cc_item = cc_forces[sid]
    cc_N = cc_item.get("axial_force", np.nan) if isinstance(
        cc_item, dict
    ) else np.nan

    gs_N = gs_forces[sid]

    # Geometric result may store a scalar or a nested record
    if isinstance(gs_N, dict):
        gs_N = gs_N.get("axial_force", gs_N.get("N", np.nan))

    try:
        cc_N = float(cc_N)
        gs_N = float(gs_N)
    except (TypeError, ValueError):
        continue

    if not np.isfinite(cc_N) or not np.isfinite(gs_N):
        continue

    abs_error = abs(cc_N - gs_N)
    denominator = max(abs(gs_N), 1e-12)
    rel_error = 100.0 * abs_error / denominator

    force_rows.append({
        "Segment ID": sid,
        "Chord axial force (N)": cc_N,
        "Geometric axial force (N)": gs_N,
        "Absolute difference (N)": abs_error,
        "Difference (%)": rel_error
    })

print("\n--- MEMBER AXIAL FORCE COMPARISON ---")

if force_rows:
    force_comparison = pd.DataFrame(force_rows)
    display(force_comparison)

    N_cc = force_comparison["Chord axial force (N)"].to_numpy()
    N_gs = force_comparison["Geometric axial force (N)"].to_numpy()

    force_norm = np.linalg.norm(N_cc - N_gs)
    force_ref = np.linalg.norm(N_gs)
    force_rel = (
        100.0 * force_norm / force_ref
        if force_ref > 1e-12 else np.nan
    )

    print(f"Compared segments: {len(force_rows)}")
    print(f"Overall axial-force difference: {force_rel:.4f}%")

    force_comparison.to_csv(
        "chord_vs_geometric_force_comparison.csv", index=False
    )
else:
    print("No matching segment axial forces found.")
    print("Check segment IDs and the stored result structures.")

# 5. Overall interpretation
print("\n--- INTERPRETATION ---")

if np.isfinite(disp_rel_error):
    if disp_rel_error < 1:
        print("Displacements: VERY CLOSE (<1% relative difference).")
    elif disp_rel_error < 5:
        print("Displacements: REASONABLY CLOSE (1–5% difference).")
    else:
        print("Displacements: NOTICEABLE DIFFERENCE (>=5%).")
else:
    print("Displacement relative difference is undefined or near zero.")

if force_rows and np.isfinite(force_rel):
    if force_rel < 1:
        print("Axial forces: VERY CLOSE (<1% overall difference).")
    elif force_rel < 5:
        print("Axial forces: REASONABLY CLOSE (1–5% overall difference).")
    else:
        print("Axial forces: NOTICEABLE DIFFERENCE (>=5%).")

print("\nNote: These percentage bands are practical comparison guides,")
print("not formal validation criteria. Check force sign conventions,")
print("matching segment IDs, and equilibrium before drawing conclusions.")

CHORD CORRECTION vs GEOMETRIC STIFFNESS
Chord correction converged: True
Geometric stiffness converged: True
Chord load factor: 1.0
Geometric load factor: 1.0

--- NODAL DISPLACEMENT COMPARISON ---
Chord displacement norm:     5.328416e-03
Geometric displacement norm: 5.327970e-03
Absolute difference norm:    1.511498e-06
Relative difference:         0.0284%
Maximum absolute DOF error:  1.373321e-06

--- MEMBER AXIAL FORCE COMPARISON ---


,Segment ID,Chord axial force (N),Geometric axial force (N),Absolute difference (N),Difference (%)
0,1,0.000000,0.000000e+00,0.000000,0.000000e+00
1,2,-5387.890600,-5.388130e+03,0.239493,4.444830e-03
2,3,3808.789245,3.808957e+03,0.167648,4.401403e-03
3,4,1695.086139,1.694843e+03,0.242888,1.433103e-02
4,5,2.903719,-2.392810e-12,2.903719,1.213518e+14
5,6,-11195.704370,-1.119553e+04,0.173792,1.552337e-03


Compared segments: 6
Overall axial-force difference: 0.0224%

--- INTERPRETATION ---
Displacements: VERY CLOSE (<1% relative difference).
Axial forces: VERY CLOSE (<1% overall difference).

Note: These percentage bands are practical comparison guides,
not formal validation criteria. Check force sign conventions,
matching segment IDs, and equilibrium before drawing conclusions.
